# 🛢️ Semana 4 · Unidad 2 — Laboratorio: El Problema del Derrame de Petróleo

---

> **Curso:** Estructuras de Datos y Algoritmos  
> **Unidad:** 2 — Tipos de Datos Abstractos  
> **Entrega:** Sube este archivo como `A2_<apellido>_<nombre>.ipynb`  
> ⚠️ El archivo `unionfind.py` debe estar en la misma carpeta que este notebook.

---

## 🎯 Objetivo

En este assignment **no implementarás** Union-Find desde cero. Usarás la librería `unionfind.py` entregada por el curso — exactamente como se hace en la industria: conoces la **API** y la usas sin ver la implementación interna.

Esto es el núcleo del **Tipo de Dato Abstracto (TDA)**:

```
┌──────────────────────────────────────────────────────┐
│              Tu código  →  clase OilSpill            │
│        usa la API sin conocer la implementación      │
└──────────────────────────┬───────────────────────────┘
                           │  from unionfind import ...
┌──────────────────────────▼───────────────────────────┐
│           unionfind.py   (caja negra 🔒)             │
│      WeightedQuickUnion — detalles ocultos           │
└──────────────────────────────────────────────────────┘
```

---

## 📖 El problema

Modelamos el océano como una **grilla N×N**. Cada celda es `True` (contaminada) o `False` (limpia).

```
     col→  0   1   2   3   4
fila ↓  ┌───┬───┬───┬───┬───┐
  0     │ . │ . │ █ │ . │ . │
        ├───┼───┼───┼───┼───┤
  1     │ . │ █ │ █ │ . │ . │
        ├───┼───┼───┼───┼───┤
  2     │ . │ █ │ . │ █ │ . │
        ├───┼───┼───┼───┼───┤
  3     │ . │ █ │ . │ . │ . │
        ├───┼───┼───┼───┼───┤
  4     │ . │ █ │ █ │ █ │ █ │
        └───┴───┴───┴───┴───┘
  █ = contaminada    . = limpia
```

Dos celdas son parte de la **misma mancha** si están conectadas horizontal o verticalmente (no diagonal).

Tu clase `OilSpill` debe responder:
1. ¿Cuántas manchas independientes hay?
2. ¿Cuántas celdas tiene la mancha más grande?
3. ¿Dos celdas están en la misma mancha?
4. ¿Una mancha toca algún borde de la grilla?

---

## 📚 API de la librería `unionfind`

Esta sección es tu **referencia completa**. No necesitas leer el código fuente de `unionfind.py`.

---

### Importación

```python
from unionfind import WeightedQuickUnion
```

---

### Constructor

```python
uf = WeightedQuickUnion(n)
```
Crea una estructura con `n` objetos identificados `0, 1, ..., n-1`.  
Al inicio hay `n` componentes (cada objeto es su propio componente).

---

### Métodos disponibles

| Método | Qué hace | Retorna | Costo |
|--------|----------|---------|-------|
| `uf.union(p, q)` | Conecta el componente de `p` con el de `q`. Si ya están conectados, no hace nada. | `None` | O(log N) |
| `uf.find(p)` | Identificador del componente al que pertenece `p`. | `int` | O(log N) |
| `uf.connected(p, q)` | `True` si `p` y `q` están en el mismo componente. | `bool` | O(log N) |
| `uf.count()` | Número de componentes actuales. | `int` | O(1) |
| `uf.component_size(p)` | Número de objetos en el mismo componente que `p`. | `int` | O(log N) |

---

### Ejemplo de uso

```python
from unionfind import WeightedQuickUnion

uf = WeightedQuickUnion(5)    # {0} {1} {2} {3} {4}  → 5 componentes

uf.union(0, 1)                # {0,1} {2} {3} {4}    → 4 componentes
uf.union(1, 2)                # {0,1,2} {3} {4}       → 3 componentes

uf.connected(0, 2)            # True  — mismo componente
uf.connected(0, 3)            # False — componentes distintos

uf.count()                    # 3
uf.component_size(0)          # 3  (componente {0,1,2})
uf.component_size(3)          # 1  (componente {3})

uf.find(0) == uf.find(2)      # True  — mismo id de componente
uf.find(0) == uf.find(3)      # False — distinto id de componente
```

---

> ⚠️ Los índices deben estar entre `0` y `n-1`. Un índice fuera de rango lanza `IndexError`.

---
## ⚙️ Celda 1 — Setup *(ejecutar primero, no modificar)*

In [ ]:
# ════════════════════════════════════════════════════════════
#  SETUP — NO MODIFICAR
# ════════════════════════════════════════════════════════════
from unionfind import WeightedQuickUnion
import unittest


def visualizar_grilla(grid, titulo="Grilla", manchas=None):
    """
    Imprime la grilla con el mismo dibujo de celdas del enunciado.

    Parámetros:
        grid (list[list[bool]]): True = contaminada, False = limpia.
        titulo (str): encabezado de la salida.
        manchas (dict | None): {(fila, col): id_mancha}, como lo retorna
            mapa_manchas(). Si se entrega, cada mancha se dibuja con su propia
            letra (A, B, C, …) y se informa su tamaño; si no, se usa █.
    """
    n = len(grid)
    manchas = manchas or {}
    letras = {}                                   # id_mancha → letra
    tamanios = {}                                 # letra → número de celdas

    print(titulo)
    print("     col→ " + "".join(f"{c:<4}" for c in range(n)))
    print("fila ↓  ┌" + "┬".join(["───"] * n) + "┐")
    for r in range(n):
        simbolos = []
        for c in range(n):
            if not grid[r][c]:
                simbolos.append(".")
            elif (r, c) in manchas:
                letra = letras.setdefault(manchas[(r, c)], chr(ord("A") + len(letras) % 26))
                tamanios[letra] = tamanios.get(letra, 0) + 1
                simbolos.append(letra)
            else:
                simbolos.append("█")
        print(f"  {r:<6}│ " + " │ ".join(simbolos) + " │")
        if r < n - 1:
            print("        ├" + "┼".join(["───"] * n) + "┤")
    print("        └" + "┴".join(["───"] * n) + "┘")
    if tamanios:
        print("  . = limpia    " + "    ".join(
            f"{letra} = {k} celda{'s' * (k != 1)}" for letra, k in tamanios.items()))
    else:
        print("  █ = contaminada    . = limpia")


# Verificación rápida de la librería
class TestLibreria(unittest.TestCase):
    """unionfind.py cargó y responde según su API."""

    def test_api_basica(self):
        """union, connected y count de WeightedQuickUnion"""
        uf = WeightedQuickUnion(5)
        uf.union(0, 1)
        uf.union(1, 2)
        self.assertTrue(uf.connected(0, 2))
        self.assertFalse(uf.connected(0, 3))
        self.assertEqual(uf.count(), 3)


resultado = unittest.main(argv=["ignorado", "TestLibreria"], exit=False, verbosity=2)

---
## 📐 Celda 2 — Mapeo grilla 2D → índice 1D

`WeightedQuickUnion` trabaja con enteros `0..N²-1`. Necesitas convertir cada celda:

```
índice = fila × N + col

N=5:  (0,0)→0  (0,1)→1  ...  (0,4)→4
      (1,0)→5  (1,1)→6  ...  (1,4)→9
      ...
      (4,4)→24
```

Los 4 vecinos de `(r,c)` son `(r-1,c)`, `(r+1,c)`, `(r,c-1)`, `(r,c+1)`.  
Solo se conectan vecinos **dentro de la grilla** y **contaminados**.

---
## 💻 Celda 3 — Tu implementación

In [ ]:
class OilSpill:
    """
    Modela un derrame de petróleo en una grilla N×N.
    Usa WeightedQuickUnion (de la librería unionfind) internamente.

    Parámetro
    ---------
    grid : list[list[bool]]
        grid[r][c] == True  →  celda (r,c) contaminada
        grid[r][c] == False →  celda (r,c) limpia
    """

    def __init__(self, grid: list):
        self.grid = grid
        self.n    = len(grid)

        # TODO 1: Crea un WeightedQuickUnion con N*N elementos
        self.uf = # TODO

        # TODO 2: Recorre todas las celdas contaminadas y únelas
        #         con sus vecinos contaminados usando self.uf.union()
        #         Usa self._idx(r, c) para obtener el índice.


    def _idx(self, r: int, c: int) -> int:
        """Convierte (fila, col) al índice del arreglo 1D."""
        return r * self.n + c


    def num_manchas(self) -> int:
        """
        TODO 3: Número de manchas (componentes de celdas contaminadas).
        Pista: set de uf.find() sobre celdas contaminadas.
        """
        pass

    def mancha_mas_grande(self) -> int:
        """
        TODO 4: Tamaño en celdas de la mancha más grande. 0 si no hay.
        Pista: usa uf.component_size() sobre celdas contaminadas.
        """
        pass

    def misma_mancha(self, r1:int, c1:int, r2:int, c2:int) -> bool:
        """
        TODO 5: True si (r1,c1) y (r2,c2) están en la misma mancha.
        False si alguna celda está limpia.
        Pista: usa uf.connected().
        """
        pass

    def llega_al_borde(self, r:int, c:int) -> bool:
        """
        TODO 6: True si la mancha de (r,c) toca algún borde.
        False si la celda está limpia.
        Pista: itera las celdas del borde y usa uf.connected().
        """
        pass

    def mapa_manchas(self) -> dict:
        """
        TODO 7: Retorna { (r,c): id_mancha } para celdas contaminadas.
        id_mancha = uf.find() de esa celda. Usado para visualizar.
        """
        pass


print("Clase OilSpill definida.")

---
## 🔍 Celda 4 — Exploración

In [ ]:
grid_ejemplo = [
    [False, False, True,  False, False],
    [False, True,  True,  False, False],
    [False, True,  False, True,  False],
    [False, True,  False, False, False],
    [False, True,  True,  True,  True ],
]
oil = OilSpill(grid_ejemplo)
print(f"Manchas            : {oil.num_manchas()}")
print(f"Mancha más grande  : {oil.mancha_mas_grande()} celdas")
print(f"¿(0,2)-(4,2) juntas? {oil.misma_mancha(0,2,4,2)}")
print(f"¿(0,2)-(2,3) juntas? {oil.misma_mancha(0,2,2,3)}")
print(f"¿(0,2) al borde?     {oil.llega_al_borde(0,2)}")
print(f"¿(2,3) al borde?     {oil.llega_al_borde(2,3)}")
print()
visualizar_grilla(grid_ejemplo, titulo="Derrame — manchas por componente", manchas=oil.mapa_manchas())

In [ ]:
# Crea tu propia grilla y experimenta
mi_grid = [
    [True,  False, False, True ],
    [True,  False, False, True ],
    [False, False, True,  True ],
    [False, True,  True,  False],
]
mi_oil = OilSpill(mi_grid)
print(f"Manchas: {mi_oil.num_manchas()}  |  Más grande: {mi_oil.mancha_mas_grande()} celdas")
print()
visualizar_grilla(mi_grid, titulo="Mi grilla", manchas=mi_oil.mapa_manchas())

---
## 🧪 Celda 5 — Tests automáticos *(no modificar)*

In [ ]:
# ════════════════════════════════════════════════════════════
#  PRUEBAS AUTOMÁTICAS (unittest) — NO MODIFICAR
# ════════════════════════════════════════════════════════════
# Cada prueba termina en ok, FAIL (resultado incorrecto) o ERROR (excepción).

GRILLA_ENUNCIADO = [
    [False, False, True,  False, False],
    [False, True,  True,  False, False],
    [False, True,  False, True,  False],
    [False, True,  False, False, False],
    [False, True,  True,  True,  True ],
]


class TestOilSpill(unittest.TestCase):
    """13 pruebas del contrato de OilSpill. La corrección usa exactamente estas."""

    def setUp(self):
        self.oil = OilSpill(GRILLA_ENUNCIADO)

    # ── grilla del enunciado: una mancha de 9 celdas y una interior de 1 ──
    def test_01_num_manchas(self):
        """T1  num_manchas: la grilla del enunciado tiene 2 manchas"""
        self.assertEqual(self.oil.num_manchas(), 2)

    def test_02_mancha_mas_grande(self):
        """T2  mancha_mas_grande: la mayor tiene 9 celdas"""
        self.assertEqual(self.oil.mancha_mas_grande(), 9)

    def test_03_misma_mancha_true(self):
        """T3  misma_mancha True: (0,2) y (4,2)"""
        self.assertTrue(self.oil.misma_mancha(0, 2, 4, 2))

    def test_04_misma_mancha_false(self):
        """T4  misma_mancha False: (0,2) y (2,3) son manchas distintas"""
        self.assertFalse(self.oil.misma_mancha(0, 2, 2, 3))

    def test_05_misma_mancha_limpia(self):
        """T5  misma_mancha False si una celda está limpia: (0,0)"""
        self.assertFalse(self.oil.misma_mancha(0, 0, 0, 2))

    def test_06_llega_al_borde_true(self):
        """T6  llega_al_borde True: (0,2) toca la fila 0"""
        self.assertTrue(self.oil.llega_al_borde(0, 2))

    def test_07_llega_al_borde_false(self):
        """T7  llega_al_borde False: (2,3) es una mancha interior"""
        self.assertFalse(self.oil.llega_al_borde(2, 3))

    def test_08_mapa_completo(self):
        """T8  mapa_manchas incluye todas las celdas contaminadas"""
        mapa = self.oil.mapa_manchas()
        for r, fila in enumerate(GRILLA_ENUNCIADO):
            for c, contaminada in enumerate(fila):
                if contaminada:
                    self.assertIn((r, c), mapa)

    def test_09_mapa_sin_limpias(self):
        """T9  mapa_manchas no incluye celdas limpias"""
        mapa = self.oil.mapa_manchas()
        for r, fila in enumerate(GRILLA_ENUNCIADO):
            for c, contaminada in enumerate(fila):
                if not contaminada:
                    self.assertNotIn((r, c), mapa)

    # ── casos borde ──
    def test_10_grilla_vacia(self):
        """T10 grilla limpia: 0 manchas y la mayor mide 0"""
        oil = OilSpill([[False] * 3 for _ in range(3)])
        self.assertEqual(oil.num_manchas(), 0)
        self.assertEqual(oil.mancha_mas_grande(), 0)

    def test_11_grilla_llena(self):
        """T11 grilla 3×3 llena: 1 mancha de 9 celdas"""
        oil = OilSpill([[True] * 3 for _ in range(3)])
        self.assertEqual(oil.num_manchas(), 1)
        self.assertEqual(oil.mancha_mas_grande(), 9)

    def test_12_esquinas_aisladas(self):
        """T12 cuatro esquinas sin vecinos: 4 manchas"""
        oil = OilSpill([[True, False, True], [False, False, False], [True, False, True]])
        self.assertEqual(oil.num_manchas(), 4)

    def test_13_simetria(self):
        """T13 misma_mancha es simétrica"""
        self.assertEqual(self.oil.misma_mancha(0, 2, 4, 2), self.oil.misma_mancha(4, 2, 0, 2))


resultado = unittest.main(argv=["ignorado", "TestOilSpill"], exit=False, verbosity=2)

---
## ✍️ Celda 6 — Preguntas de reflexión

**Pregunta 1:** Usaste `WeightedQuickUnion` sin ver su código. ¿Qué ventaja tiene ese enfoque? ¿Qué pasaría si la cátedra cambiara la implementación interna pero mantuviera la misma API?

> *Tu respuesta aquí...*

**Pregunta 2:** `num_manchas()` no puede usar `self.uf.count()` directamente. ¿Por qué? ¿Qué contaría de más?

> *Tu respuesta aquí...*

**Pregunta 3:** ¿Cuál es la complejidad de `__init__` en función de N? Considera el recorrido y el costo de cada `union()`. Justifica.

> *Tu respuesta aquí...*

---
## ✅ Celda 7 — Checklist de entrega

- [ ] `Kernel → Restart & Run All` sin errores
- [ ] `TestOilSpill` termina en `OK` (13 pruebas)
- [ ] Las 3 preguntas respondidas
- [ ] Archivo nombrado `A2_<apellido>_<nombre>.ipynb`
- [ ] `unionfind.py` en la misma carpeta al momento de ejecutar

**Nombre:** ___________________________  
**Fecha:** ___________________________